# Dataset Inspector

Use this notebook to load an arbitrary offline dataset (`.npy`) and inspect trajectories interactively.

This notebook provides:
- An interactive trajectory/state viewer with two sliders
- A trajectory statistics summary table

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import ipywidgets as widgets
from IPython.display import display


def load_dataset(path: str | Path) -> dict:
    path = Path(path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    data = np.load(path, allow_pickle=True)
    if hasattr(data, "item"):
        data = data.item()
    if not isinstance(data, dict):
        raise TypeError("Expected dataset to be a dict saved in .npy")
    return data


def extract_trajectories(dataset: dict) -> list[dict]:
    if "trajectories" in dataset:
        trajectories = dataset["trajectories"]
        if isinstance(trajectories, np.ndarray):
            return [dict(t) for t in trajectories.tolist()]
        return [dict(t) for t in trajectories]
    raise KeyError("Dataset does not contain a 'trajectories' field.")


def infer_grid_size(dataset: dict, trajectories: list[dict]) -> int:
    cfg = dataset.get("config", {})
    if isinstance(cfg, dict) and "grid_size" in cfg:
        return int(cfg["grid_size"])

    for traj in trajectories:
        obs = np.asarray(traj.get("observations", []))
        if obs.ndim >= 3:
            return int(obs.shape[-1])
        if obs.ndim == 2 and obs.shape[1] > 0:
            side = int(np.sqrt(obs.shape[1]))
            if side * side == obs.shape[1]:
                return side

    raise ValueError("Could not infer grid size from dataset.")


def get_states(traj: dict, grid_size: int) -> np.ndarray:
    obs = np.asarray(traj.get("observations", []))
    if obs.ndim == 3:
        return obs
    if obs.ndim == 2:
        return obs.reshape(obs.shape[0], grid_size, grid_size)
    raise ValueError("Unsupported observation shape in trajectory.")


# A readable discrete palette for integer entity ids
GRID_COLORS = [
    "#000000",  # 0
    "#f5f5f5",  # 1
    "#1f77b4",  # 2
    "#ff7f0e",  # 3
    "#2ca02c",  # 4
    "#d62728",  # 5
    "#9467bd",  # 6
    "#8c564b",  # 7
    "#e377c2",  # 8
    "#7f7f7f",  # 9
    "#bcbd22",  # 10
    "#17becf",  # 11
]
CMAP = ListedColormap(GRID_COLORS)
NORM = BoundaryNorm(np.arange(-0.5, len(GRID_COLORS) + 0.5, 1.0), CMAP.N)

In [2]:
# Set this to any dataset you want to inspect
# Use a path relative to this notebook (notebooks/), or an absolute path.
DATASET_PATH = "../data/expert_variable_special_6x6_6boxes_10traj_parallel10_fixed100.npy"

dataset = load_dataset(DATASET_PATH)
trajectories = extract_trajectories(dataset)
GRID_SIZE = infer_grid_size(dataset, trajectories)

print(f"Loaded: {Path(DATASET_PATH).resolve()}")
print(f"Trajectories: {len(trajectories)}")
print(f"Grid size: {GRID_SIZE}x{GRID_SIZE}")

Loaded: /home/mbortkie/repos/golden-standard/data/expert_variable_special_6x6_6boxes_10traj_parallel10_fixed100.npy
Trajectories: 10
Grid size: 6x6


In [3]:
if len(trajectories) == 0:
    raise ValueError("No trajectories found in dataset.")

traj_slider = widgets.IntSlider(value=0, min=0, max=len(trajectories) - 1, step=1, description="trajectory")
step_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="state")
out = widgets.Output()


def update_step_slider(*_):
    states = get_states(trajectories[traj_slider.value], GRID_SIZE)
    max_step = max(0, int(states.shape[0]) - 1)
    step_slider.max = max_step
    if step_slider.value > max_step:
        step_slider.value = max_step


def render_state(*_):
    states = get_states(trajectories[traj_slider.value], GRID_SIZE)
    step = int(step_slider.value)
    grid = states[step]

    traj = trajectories[traj_slider.value]
    success = traj.get("success", None)
    executed_steps = traj.get("executed_steps", states.shape[0])

    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(grid, cmap=CMAP, norm=NORM, interpolation="nearest")
        ax.set_xticks(np.arange(-0.5, GRID_SIZE, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, GRID_SIZE, 1), minor=True)
        ax.grid(which="minor", color="black", linewidth=0.5)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        ax.set_title(
            f"traj={traj_slider.value} | state={step}/{states.shape[0] - 1} | steps={executed_steps} | success={success}"
        )
        plt.show()


traj_slider.observe(update_step_slider, names="value")
traj_slider.observe(render_state, names="value")
step_slider.observe(render_state, names="value")

update_step_slider()
render_state()
display(widgets.VBox([traj_slider, step_slider, out]))

In [4]:
rows = []
for i, traj in enumerate(trajectories):
    obs = np.asarray(traj.get("observations", []))
    rewards = np.asarray(traj.get("rewards", []), dtype=np.float32)

    executed_steps = int(traj.get("executed_steps", obs.shape[0] if obs.ndim > 0 else 0))
    planned_actions = int(traj.get("planned_actions", len(traj.get("actions", []))))
    success = bool(traj.get("success", False))

    if "discounted_return" in traj:
        discounted_return = float(traj["discounted_return"])
    else:
        discounted_return = 0.0
        for r in rewards[::-1]:
            discounted_return = discounted_return * 0.99 + float(r)

    rows.append(
        {
            "trajectory": i,
            "executed_steps": executed_steps,
            "planned_actions": planned_actions,
            "success": success,
            "discounted_return": discounted_return,
            "reward_sum": float(rewards.sum()) if rewards.size else 0.0,
        }
    )

traj_df = pd.DataFrame(rows)
display(traj_df)

summary_df = pd.DataFrame(
    {
        "metric": [
            "num_trajectories",
            "success_rate",
            "executed_steps_min",
            "executed_steps_mean",
            "executed_steps_max",
            "planned_actions_mean",
            "discounted_return_mean",
            "discounted_return_min",
            "discounted_return_max",
            "total_transitions",
        ],
        "value": [
            int(len(traj_df)),
            float(traj_df["success"].mean()) if len(traj_df) else np.nan,
            int(traj_df["executed_steps"].min()) if len(traj_df) else np.nan,
            float(traj_df["executed_steps"].mean()) if len(traj_df) else np.nan,
            int(traj_df["executed_steps"].max()) if len(traj_df) else np.nan,
            float(traj_df["planned_actions"].mean()) if len(traj_df) else np.nan,
            float(traj_df["discounted_return"].mean()) if len(traj_df) else np.nan,
            float(traj_df["discounted_return"].min()) if len(traj_df) else np.nan,
            float(traj_df["discounted_return"].max()) if len(traj_df) else np.nan,
            int(traj_df["executed_steps"].sum()) if len(traj_df) else 0,
        ],
    }
)

display(summary_df)

,trajectory,executed_steps,planned_actions,success,discounted_return,reward_sum
0,0,100,85,True,0.429889,1.0
1,1,100,73,True,0.484991,1.0
2,2,100,83,True,0.438618,1.0
3,3,100,73,True,0.484991,1.0
4,4,100,80,True,0.452044,1.0
5,5,100,83,True,0.438618,1.0
6,6,100,81,True,0.447523,1.0
7,7,100,76,True,0.470587,1.0
8,8,100,75,True,0.475340,1.0
9,9,100,71,True,0.494839,1.0


,metric,value
0,num_trajectories,10.000000
1,success_rate,1.000000
2,executed_steps_min,100.000000
3,executed_steps_mean,100.000000
4,executed_steps_max,100.000000
5,planned_actions_mean,78.000000
6,discounted_return_mean,0.461744
7,discounted_return_min,0.429889
8,discounted_return_max,0.494839
9,total_transitions,1000.000000
